# Preliminary MG PTDF heatmap

A PTDF measures the change in flow on a branch for a **1 MW injection at a candidate point of interconnection (POI)** paired with a **1 MW pro rata reduction in other online generation**. Positive and negative signs follow the PWF branch direction. The heatmap uses the full electrical network from `hosting_capacity/hc_engine.py`, then displays the 18 monitored main-grid lines with the largest absolute sensitivities across the selected POIs.

This is a **DC sensitivity illustration**, not a hosting-capacity result, overload test, N-1 finding, or proof that a site has suitable solar or wind resource. The grid input CSVs are private and must be restored locally before running. This notebook does not add them to GitHub.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'hosting_capacity' / 'hc_engine.py').is_file()), None)
if ROOT is None:
    raise FileNotFoundError('Run this notebook inside a checkout containing hosting_capacity/hc_engine.py')
INPUTS = ROOT / 'MG_Step2_Congestion_Relief' / 'MG_Nodal_Connection_Optimization' / 'inputs'
needed = [INPUTS / 'pwf_2035_buses.csv', INPUTS / 'pwf_2035_branches.csv']
missing = [p.name for p in needed if not p.is_file()]
if missing:
    raise FileNotFoundError(f'Restore the private PWF inputs locally: {missing}')
sys.path.insert(0, str(ROOT / 'hosting_capacity'))
from hc_engine import Network
FIGURES = ROOT / 'planning_study' / 'notebooks' / 'figures'
print('Repository:', ROOT)


In [ ]:
net = Network(inputs=INPUTS)
poi_file = ROOT / 'hosting_capacity' / 'results' / 'poi_hosting_capacity.csv'
poi = pd.read_csv(poi_file).sort_values('hc_P_prorata_mw', ascending=False).drop_duplicates('bus_id').head(10).copy()
assert len(poi) == 10 and poi['bus_id'].isin(net.bus_ids).all()
withdrawal, headroom_mw, label = net.balancing('P_prorata')
assert np.isclose(withdrawal.sum(), 1.0)
print(f'{len(net.B):,} buses; {len(net.L):,} branches; {net.n_lines:,} monitored main-grid lines')
print('Withdrawal:', label)
print(f'Available generation backdown in the reference snapshot: {headroom_mw:,.0f} MW')
display(poi[['station', 'kv', 'hc_P_prorata_mw']].rename(columns={'hc_P_prorata_mw': 'Separate N-0/N-1 hosting result (MW)'}).round(0))


In [ ]:
line_idx = net.monitored[net.kind_mon == 'line']
poi_idx = np.array([net.idx[int(bus)] for bus in poi['bus_id']])
# PTDF for an injection at each POI, minus 1 MW of pro rata generation backdown.
sens = net.PTDF[np.ix_(line_idx, poi_idx)] - (net.PTDF[line_idx] @ withdrawal)[:, None]
assert sens.shape == (len(line_idx), len(poi)) and np.isfinite(sens).all()
chosen = np.argsort(np.abs(sens).max(axis=1))[-18:][::-1]
line_idx, matrix = line_idx[chosen], sens[chosen]

def bus_name(i):
    for column in ('name', 'bus_name', 'nome'):
        if column in net.B.columns:
            return str(net.B.iloc[i][column]).strip()
    return str(net.bus_ids[i])

row_labels = [f'{int(i)}: {bus_name(net.f[i])} → {bus_name(net.t[i])}' for i in line_idx]
col_labels = [f'{r.station} ({int(r.kv)} kV)' for r in poi.itertuples()]
vmax = max(float(np.abs(matrix).max()), 1e-6)
fig, ax = plt.subplots(figsize=(13, 8))
im = ax.imshow(matrix, aspect='auto', cmap='RdBu_r', vmin=-vmax, vmax=vmax)
ax.set_xticks(np.arange(len(col_labels)), col_labels, rotation=45, ha='right')
ax.set_yticks(np.arange(len(row_labels)), row_labels)
ax.set_xlabel('Candidate point of interconnection (1 MW injection)')
ax.set_ylabel('Main-grid line (PWF branch direction)')
ax.set_title('MG transfer PTDF: pro rata generation backdown')
fig.colorbar(im, ax=ax, label='Flow change (MW per 1 MW injection)')
fig.tight_layout()
FIGURES.mkdir(exist_ok=True)
out = FIGURES / 'preliminary_ptdf_heatmap.png'
fig.savefig(out, dpi=180, bbox_inches='tight')
plt.show()
print('Saved:', out.relative_to(ROOT))


## Reading the heatmap

A cell of `+0.30` means that this balanced 1 MW transfer would raise flow by 0.30 MW in the listed line direction. A negative value lowers flow in that direction; **neither sign alone says whether the line is relieved or overloaded**. Thermal ratings, starting flows, contingencies, simultaneous additions, and AC voltage behavior are separate checks in the team's deliverability workflow.

The top ten POIs come from the saved hosting-capacity table solely to keep the plot readable. Their hosting values are computed separately under N-0/N-1 constraints; they are **not** derived from this heatmap. The 2035 PWF network snapshot is the electrical basis, including for discussion of 2050 scenarios.